<CENTER>
    <a href="http://opendata.atlas.cern" class="icons"><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/images/ATLASOD.gif?raw=1" style="width:50%"></a>
</CENTER>

# Searching for the Higgs boson in the $H\to\gamma\gamma$ channel using C++ 

## What is the Higgs boson?

The Higgs boson is a fundamental particle predicted by the [Standard Model](https://cds.cern.ch/record/2759492/files/Standard%20Model%20-%20ATLAS%20Physics%20Cheat%20Sheet.pdf ). It is a manifestation of the Higgs field, which gives mass to fundamental particles. This means that for a Higgs boson to arise, it requires a high amount of energy concentrated into a single process, exciting the Higgs field through high energy exchanges. This makes the Higgs boson an incredibly rare particle, as the energies required to cause a sufficient excitation are rarely reached by single particles. At the LHC, a Higgs boson is produced approximately once every 10 billion collisions, making it very challenging to detect.

Despite this tiny fraction, years of data collection led to the discovery of the Higgs boson in 2012 by the CMS and ATLAS experiments at CERN. In this tutorial, we will utilize a similar method used by the 2012 discovery to rediscover the Higgs boson using C++!

## Detecting the Higgs Boson

The Higgs boson can be generated through various mechanisms. In particle physics, we use [Feynman diagrams](https://cds.cern.ch/record/2759490/files/Feynman%20Diagrams%20-%20ATLAS%20Cheat%20Sheet.pdf) to illustrate these production modes. These diagrams help us visualize particle interactions and serve as essential tools for computations. For additional details on Feynman diagrams, see this link.

There are four primary production modes for the Higgs boson, each represented by their own Feynman diagram:

<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/images/Figures_FeynmanHprod.png?raw=1" style="width:50%"></CENTER>

- Gluon-gluon fusion (top left) 
- Vector boson fusion (top right) 
- Vector boson bremsstrahlung (bottom left) 
- Top-antitop fusion (bottom right)

The Higgs boson has an extremely brief lifetime, approximately $10^{-22} s$. It decays almost immediately after it is produced, making direct detection of the particle impossible. However, by using predictions made by the Standard Model, we can catalog all of the possible ways a Higgs boson can decay, and understand the probability and kinematics of each decay channel. These decay channels help us identify the presence of the Higgs boson without ever detecting it directly. In this notebook, we will focus on analyzing one specific decay channel: the decay of the Higgs boson into two photons, denoted as $H \to \gamma\gamma$.

<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/13-TeV-examples/uproot_python/images/feynman_diagrams/Hyy_feynman.png?raw=1" style="width:40%"></CENTER>

We refer to this as our desired *[signal](https://cds.cern.ch/record/2800577/files/Signal%20and%20Background%20Physics%20Cheat%20Sheet.pdf)*. In this analysis, we aim to identify collisions that produce two photons, which would indicate the presence of a Higgs boson. However, many processes found at the LHC which do not include the Higgs boson also produce photons, which serve as *background* photon contributors.

Backgrounds are classified into two categories: reducible and irreducible. Reducible backgrounds can be significantly minimized using experimental techniques like data cuts, particle identification, and isolation criteria. For example, a reducible background might involve events where a jet is misidentified as a photon. By applying strict criteria that filters out detected "photons" which behave more like jets, this background can be reduced, giving us a sample with more accurate photon information.

On the other hand, irreducible backgrounds cannot be easily distinguished from the signal because they involve the same final states or processes that the signal would produce. In our $H \to\gamma\gamma$ channel, an irreducible background would be the direct production of two photons from other Standard Model processes, such as quark-antiquark annihilation. These events are fundamentally indistinguishable from the signal events based on [final state particles](https://cds.cern.ch/record/2951043/files/Inside%20a%20Collision.pdf) alone.

To address this, we can consider the total [invariant mass](https://cds.cern.ch/record/2919871/files/4-vectors%20and%20invariant%20mass%20cheat%20sheet.pdf) of the photon products. The Higgs boson has an invariant rest mass of ~125 GeV. By the [conservation of energy and momentum](https://cds.cern.ch/record/2759491/files/Conservation%20Laws%20-%20ATLAS%20Physics%20Cheat%20Sheet.pdf), the invariant mass of the two photons produced by our signal process will add up to approximately 125 GeV, while the invariant mass of the background photons are not constrained by this and can have any diphoton mass. The aim of our study will be to plot the diphoton masses of our signal and background processes, and look for a peak around 125 GeV, signifying a unique diphoton contribution from Higgs boson decay!

### Running a Jupyter notebook
A Jupyter notebook consists of cell blocks, each containing lines of Python code. Each cell can be run independently of each other, yielding respective outputs below the cells. Conventionally,cells are run in order from top to bottom.

- To run the whole notebook, in the top menu click Cell $\to$ Run All.
- To propagate a change you've made to a piece of code, click Cell $\to$ Run All Below.
- You can also run a single code cell, by clicking Cell $\to$ Run Cells, or using the keyboard shortcut Shift+Enter.

For more information, refer to [How To Use Jupyter Notebooks](https://www.codecademy.com/article/how-to-use-jupyter-notebooks).

### Setup

We'll start by including some standard C++ and ROOT libraries which we'll use to analyze the data. We'll also import jsroot to allow us to use interactive plots for our analysis histograms, rather than flat PNGs.

In [ ]:
// Creates a TChain to be used by the Analysis.C class
%jsroot on
#include <TChain.h>
#include <vector>
#include <TFile.h>
#include <iostream>
#include <string>
#include <stdio.h>
#include <chrono>

Next, we'll load our data. Here, we'll use the root (or HTTPS) link for the ATLAS Open Data release. In this analysis, we'll be using real collision data gathered at the LHC! Since this analysis focuses on a very unique mass range that isn't very common, we will have a high signal-to-background ratio, meaning a Higgs mass peak will be very obvious. 

The following code cell will load ATLAS collision data files from the CERN Open Data portal, combine them into a single format, and return to us how many events we have to work with. Because we would like to use more than one ROOT input file, the best option is to use a TChain object. This allows to "chain" several samples into a single structure that we can later loop over. 

In [ ]:
//In case xrootd is not available where you run this notebook, you can use the https address
//TString path = "https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/";
TString path = "root://eospublic.cern.ch:1094//eos/opendata/atlas/rucio/opendata/";

TChain* fChain = new TChain("analysis");

TString prefix = "ODEO_FEB2025_v0";
TString skim = "GamGam";

vector<TString> samples = { "data15_periodD", "data15_periodE", "data15_periodF", "data15_periodG", 
                            "data15_periodH", "data15_periodJ", "data16_periodA", "data16_periodB", 
                            "data16_periodC", "data16_periodD", "data16_periodE", "data16_periodF", 
                            "data16_periodG", "data16_periodI", "data16_periodK", "data16_periodL"
                           };

cout << "Working on " << samples.size() << " samples" << endl;

// Set the number of samples you want to include in the analysis (16 data samples in total)
int max_samples = samples.size();

for(int ii=0; ii<max_samples; ii++){
    TString name_sample = samples.at(ii);
    TString name_root_file = path+prefix+"_"+skim+"_"+name_sample+"."+skim+".root";
    cout << "Adding file: " << name_root_file << endl;
    fChain->Add(name_root_file);
}

int nentries = (Int_t)fChain->GetEntries();
cout << "Number of entries: " << nentries << endl;

Now that we have our data loaded, we'll load each variable we're going to use in our analysis in the code below! Each variable is described in the cell below, as well as their purpose/meaning as needed. 

In [ ]:
Int_t  photon_n;  // Number of preselected photons                                                                                                             

ROOT::VecOps::RVec<float> *photon_pt = nullptr;  // Transverse momentum of the photon                                                                                    
ROOT::VecOps::RVec<float> *photon_eta = nullptr;  // Pseudorapidity of the photon                                                                                        
ROOT::VecOps::RVec<float> *photon_phi = nullptr;  // Azimuthal angle of the photon                                                                                       
ROOT::VecOps::RVec<float> *photon_e = nullptr;  // Energy of the photon                                                                                                  
ROOT::VecOps::RVec<bool> *photon_isTightID = nullptr; // Does the photon pass tight ID requirements?
ROOT::VecOps::RVec<bool> *photon_isTightIso = nullptr; // Does the photon pass tight isolation requirements?

Bool_t trigP; // Did the event pass the photon trigger?
ROOT::VecOps::RVec<float> *photon_ptcone20 = nullptr; // The photon isolation estimate (track momentum in a cone)

Now that we've loaded our data and our variables, we'll load each value of the variables we loaded for all of the events! This lets us cut, filter, and manipulate our data for analysis and histogramming later in the notebook. 

In [ ]:
fChain->SetBranchAddress("photon_pt",        &photon_pt);
fChain->SetBranchAddress("photon_n",         &photon_n);

fChain->SetBranchAddress("photon_eta",       &photon_eta);
fChain->SetBranchAddress("photon_phi",       &photon_phi);
fChain->SetBranchAddress("photon_e",         &photon_e);
fChain->SetBranchAddress("photon_isTightID", &photon_isTightID);
fChain->SetBranchAddress("photon_isTightIso", &photon_isTightIso);

fChain->SetBranchAddress("trigP",            &trigP);
fChain->SetBranchAddress("photon_ptcone20",  &photon_ptcone20);

For this example, we'll use a histogram to represent our data and allow us to compare differences between non-Higgs and Standard Model Higgs boson behavior. The following cell will create our histogram, and set a few basic design choices for later. 

In [ ]:
// Invariant mass histograms definition
TH1F *h_M_Hyy = new TH1F("h_M_Hyy","Diphoton invariant-mass ; M_{#gamma#gamma} [GeV] ; Events / bin", 60, 100, 160);

// Histogram style
h_M_Hyy->SetMarkerSize(2.0);
h_M_Hyy->SetLineColor(kBlue);
h_M_Hyy->SetFillColor(kBlue-10);

### Selection Criteria

Our signal process is the production of two photons from the decay of a Higgs boson. Therefore, we must look at photons produced in pairs if we ever hope to pick out two produced from the same source with such a specific total mass. To do this, we will use a series of strict selection criteria to ensure that all of the photons we observe in this analysis were produced in pairs. This allows us to fairly sum their masses and draw a baseline prediction, wherein we look for a deviation caused by an extra decay causing diphoton pairs around a specific mass (our signal process). 

The event selection criteria we will use in this analysis are:
- Transverse Momentum Cut: photon 1 has $p_T > 50$ GeV and photon 2 has $p_T > 30$ GeV. This cuts out low-energy photon systems which crowd our data and could include non-photon objects.
- Calorimeter Isolation Cut: This variable is a complex measurement of how well an event is characterized. In calorimeter isolation, a low percentage means that the particle identified makes up a majority of the energy deposited into that calorimeter at that moment. For this, we set the maximum at 5.5%, meaning that for each photon, 5.5% or less of the energy in the calorimeter that took that reading is not from the photon identified, giving us clean isolated photons.
- Pseudorapidity cut: Pseudorapidity is the measure of how parallel a particle is moving towards the beam, represented by $|\eta|$. For this analysis, we set the maximum pseudorapidity to a small value of $|\eta| < 2.37$ such that we only get photons which are detected in the primary barrel. This helps reduce noise from jets and other misidentified contributions. 

The following cell completes the primary cutflow/analysis step of this notebook, so please allow 2-5 minutes for it to complete. Once this is done, we'll be able to see how many events passed our criteria, and move on to plotting. 

In [ ]:
auto start_time = chrono::steady_clock::now(); //Start the clock

int nbytes = 0;
int accepted_events = 0;
std::cout << "Number of total events: " << nentries << endl;

// Set the fraction of events to be analyzed
float fraction = 0.1; // 10% of total events is used in order to reduce the time processing of this analysis example
float nevents_to_process = fraction*nentries;  

for(Long64_t i = 0; i < nevents_to_process; i++){
    nbytes = fChain->GetEntry(i);
    // printing the evolution in number of events
    if(i%1000000==0) cout << "Number of events analyzed: " << i << endl;
    if(photon_isTightID->at(0)==true && photon_isTightID->at(1)==true){
        // Cut: pT cut - photon 1 has pT > 50 GeV and photon 2 has pT > 30 GeV
        if(photon_pt->at(0) > 50. && photon_pt->at(1) > 30.){
            // Only the events where the calorimeter isolation is less than 5.5% are kept
            if(photon_ptcone20->at(0)/photon_pt->at(0) < 0.055 && photon_ptcone20->at(1)/photon_pt->at(1) < 0.055){
                // Cut on the pseudorapidity in barrel/end-cap transition region
                if((TMath::Abs(photon_eta->at(0)) < 2.37) && 
                   (TMath::Abs(photon_eta->at(0)) < 1.37 || TMath::Abs(photon_eta->at(0)) > 1.52)){
                    if((TMath::Abs(photon_eta->at(1)) < 2.37) && 
                       (TMath::Abs(photon_eta->at(1)) < 1.37 || TMath::Abs(photon_eta->at(1)) > 1.52)){
                        // TLorentzVector definitions
                        TLorentzVector Photon_1 = TLorentzVector();
                        TLorentzVector Photon_2 = TLorentzVector();

                        Photon_1.SetPtEtaPhiE(photon_pt->at(0), photon_eta->at(0), photon_phi->at(0), photon_e->at(0));
                        Photon_2.SetPtEtaPhiE(photon_pt->at(1), photon_eta->at(1), photon_phi->at(1), photon_e->at(1));
                        
                        // Calculation of the Invariant Mass using TLorentz vectors
                        TLorentzVector Photon_12 = Photon_1 + Photon_2;
                        float inv_mass_Hyy = Photon_12.M();

                        // Cut on null diphoton invariant mass
                        if(inv_mass_Hyy == 0) continue;
                        // Cut on diphoton invariant mass based isolation. Only the events where 
                        // the invididual photon invariant mass based isolation is larger than 35% are kept
                        if((photon_pt->at(0)/inv_mass_Hyy > 0.35) && (photon_pt->at(1)/inv_mass_Hyy > 0.35)){
                            // Filling with the mass of the gamma-gamma system
                            h_M_Hyy->Fill(inv_mass_Hyy);
                            accepted_events+=1;
                        }
                    }
                }
            }
        }
    }
    
}

// Final message after analysis
std::cout << "* Analysed a total of: " << nevents_to_process << " events." << std::endl;
std::cout << "* Number of events passing the selection criteria: " << accepted_events << " in this sample." << std::endl;

## Plotting 

Now that we've cut our data up and prepared it, we're ready to plot! We've got a few things we have to do modelling-wise before we can see our plot, so we'll run through everything quickly before we analyze our final product!

### Fitting

To make an effective comparison, we'll use predictive modelling to show what our signal and background *should* look like, and overlay our data to see what matches up. First, we'll create a signal model line, in the form of a gaussian distribution centered around 125 GeV. This represents the behaviour of our $H \to\gamma\gamma$ produced photons, and when fit to the data, will give us a small peak at 125 GeV, demonstrating what signal contributions look like. Then, we'll create the background model, which is essentially a straight line through the data with no peaks or deviations at any mass. 

These will be fit over the data line, such that they take the general shape of the data, deviating in their peaks and features. These model lines allow us to compare our data to both signal and background cases, and will ultimately let us identify the presence of the Higgs boson. 

Following this, we'll import the histogram we created earlier, load our processed data into it, and input more stylistic choices like a legend and labels. The following histogram will show the probability of finding dilepton pairs at each energy level measured, along with a non-Higgs background prediction line, and a Higgs-boson-inclusive prediction line. We'll also have a small box below the main histogram, which will act as a "zoom in", showing us precise deviations from the background model, giving us more detail and uncertainty information around the 125 GeV range. 

Before running the following cell and reading the graph, make some predictions: which model do you think the data will obey? If there is a peak in the data, how high will it be? What will the rest of the histogram look like? How do you expect the general distribution of diphoton masses to look like, and do you expect any other deviations than the Higgs contribution?

In [ ]:
//*****************************************************************************************************************
// Fitting part
//*****************************************************************************************************************

using namespace RooFit;

// Convert histogram into RooDataHist
RooRealVar x("m_yy", "di-photon invariant mass m_{#gamma#gamma} [GeV]", 100, 160);
RooDataHist data_myy("data_myy", "data_myy", RooArgList(x), h_M_Hyy);

// --- Define signal model (Gaussian) ---
RooRealVar mean("mean", "mean of gauss", 125, 120, 130);
RooRealVar sigma("sigma", "width of gauss", 2.0, 0.5, 5.0);
RooGaussian gauss("gauss", "gaussian PDF", x, mean, sigma);

// --- Define background model (4th order polynomial) ---
RooRealVar a0("a0", "a0", 0.0, -1, 1);
RooRealVar a1("a1", "a1", 0.0, -1, 1);
RooRealVar a2("a2", "a2", 0.0, -1, 1);
RooRealVar a3("a3", "a3", 0.0, -1, 1);
RooRealVar a4("a4", "a4", 0.0, -1, 1);
RooChebychev bkg("bkg", "Chebychev background", x, RooArgList(a0,a1,a2,a3,a4));

// --- Define coefficients (yields) ---
RooRealVar nsig("nsig", "signal yield", h_M_Hyy->Integral()/20., 0., h_M_Hyy->Integral());
RooRealVar nbkg("nbkg", "background yield", h_M_Hyy->Integral(), 0., h_M_Hyy->Integral());

// --- Total model ---
RooAddPdf model("model", "sig + bkg", RooArgList(gauss, bkg), RooArgList(nsig, nbkg));

// --- Perform the fit ---
model.fitTo(data_myy, RooFit::Extended(true), RooFit::Save(), RooFit::PrintLevel(-1));

std::cout << "mean: " << mean.getVal() << endl;
std::cout << "std deviation: " << sigma.getVal() << endl;
std::cout << "polynomic function (using the Chebyshev polynomials basis):" << endl;
std::cout << a0.getVal() << "+" << a1.getVal() << "*T_1(x)+" << a2.getVal() << "*T_2(x)+" << a3.getVal() << "*T_3(x)+" << a4.getVal() << "*T_4(x)" << endl; 

// =======================
// Main Pad (Data + Fit)
// =======================

TCanvas* canvas = new TCanvas("canvas", "canvas", 800, 600);

TPad* mainPad = new TPad("mainPad", "mainPad", 0, 0.3, 1, 1.0);
mainPad->SetBottomMargin(0);
mainPad->Draw();
mainPad->cd();

// Get histogram representation of data and background
// Ensure both histograms use the same binning

int nbins = h_M_Hyy->GetNbinsX();
double xmin = h_M_Hyy->GetXaxis()->GetXmin();
double xmax = h_M_Hyy->GetXaxis()->GetXmax();

TH1* h_data = data_myy.createHistogram("h_data", x, RooFit::Binning(nbins, xmin, xmax));
TH1* h_bkg  = bkg.createHistogram("h_bkg", x, RooFit::Binning(nbins, xmin, xmax));

std::cout << "nbins h_data: " << h_data->GetNbinsX() << endl;
std::cout << "nbins h_bkg: " << h_bkg->GetNbinsX() << endl;

std::cout << "===================================================================" << endl;
std::cout << "integral h_bkg: " << h_bkg->Integral() << endl;
std::cout << "nbkg expected: " << nbkg << endl;

// Normalize background histogram to expected number of events
h_bkg->Scale(nbkg.getVal() / h_bkg->Integral());

// Create residual histogram (data - bkg)
TH1F* resid = new TH1F("residual","Diphoton invariant-mass ; Invariant Mass m_{yy} [GeV] ; events",60,100,160);

for(int i = 1; i <= h_data->GetNbinsX(); i++) {
    float data_value = h_data->GetBinContent(i);
    float bkg_value = h_bkg->GetBinContent(i);

    // residual value
    float res_value  = data_value - bkg_value;
    resid->SetBinContent(i, res_value);
}

// --- Plot results ---
RooPlot* frame = x.frame();
data_myy.plotOn(frame, RooFit::MarkerStyle(20), RooFit::MarkerSize(0.5));
model.plotOn(frame, RooFit::Name("sig_bkg_curve"), RooFit::LineWidth(2), 
             RooFit::LineColor(kBlue));
model.plotOn(frame, RooFit::Components("bkg"), RooFit::LineWidth(2), RooFit::LineStyle(kDashed), 
             RooFit::LineColor(kRed), RooFit::Name("bkg_curve"));
model.plotOn(frame, RooFit::Components("gauss"), RooFit::LineStyle(kDotted), 
             RooFit::LineColor(kBlue), RooFit::Name("gaussian_curve"));

frame->SetTitle("Di-photon invariant mass m_{#gamma#gamma}");
frame->GetYaxis()->SetTitle("Events / bin");
frame->GetXaxis()->SetTitleSize(0.05);   // size of x-axis title
frame->GetYaxis()->SetTitleSize(0.05);   // size of y-axis title
frame->GetXaxis()->SetLabelSize(0.03);   // size of x-axis labels
frame->GetYaxis()->SetLabelSize(0.03);   // size of y-axis labels
frame->GetXaxis()->SetTitleFont(42);     // Helvetica (ROOT default)
frame->GetYaxis()->SetTitleFont(42);

frame->Draw();

canvas->cd();

// Legend
RooCurve* sig_bkg_curve = (RooCurve*) frame->findObject("sig_bkg_curve");
RooCurve* bkg_curve = (RooCurve*) frame->findObject("bkg_curve");

TLegend* legend = new TLegend(0.55, 0.75, 0.85, 0.90);
legend->AddEntry(frame, "Data", "lep");
legend->AddEntry(sig_bkg_curve, "Sig+Bkg Fit (m_{H}=125 GeV)", "l");
legend->AddEntry(bkg_curve, "Bkg (4th order polynomial)", "l");
legend->Draw();

// Labels
TLatex* latex = new TLatex();
latex->SetTextSize(0.03);
latex->DrawLatexNDC(0.60, 0.70, "ATLAS Open Data");
latex->DrawLatexNDC(0.60, 0.65, "#sqrt{s}=13 TeV, #int L dt = 36 fb^{-1}");
latex->DrawLatexNDC(0.60, 0.60, "H #rightarrow #gamma#gamma");

// =======================
// Residual Pad (Data-Bkg vs Fit-Bkg)
// =======================
TPad* residPad = new TPad("residPad", "residPad", 0, 0.05, 1, 0.3);
residPad->SetTopMargin(0);
residPad->SetBottomMargin(0.3);
residPad->Draw();
residPad->cd();

// Style
resid->SetMarkerStyle(20);
resid->SetMarkerSize(0.5);
resid->SetLineColor(kBlack);
resid->GetYaxis()->SetTitle("Events - Bkg");
resid->GetYaxis()->SetTitleSize(0.10);
resid->GetYaxis()->SetTitleOffset(0.5);
resid->GetYaxis()->SetLabelSize(0.10);
resid->GetXaxis()->SetTitle("di-photon invariant mass m_{#gamma#gamma} [GeV]");
resid->GetXaxis()->SetTitleSize(0.10);
resid->GetXaxis()->SetLabelSize(0.10);
resid->SetStats(0);

// Draw residuals (data - bkg)
resid->Draw("E1");

// Zero line
TLine* zeroLine = new TLine(100, 0, 160, 0);
zeroLine->SetLineColor(kRed);
zeroLine->SetLineStyle(2);
zeroLine->Draw("same");

RooCurve* gaussian_curve = (RooCurve*) frame->findObject("gaussian_curve");
gaussian_curve->Draw("same");

canvas->Draw();

auto end_time = chrono::steady_clock::now();
auto elapsed = chrono::duration_cast<chrono::seconds>(end_time - start_time).count();
cout << "Time taken: " << elapsed/60. << " min." << endl;

// If you want a copy of your plot locally, uncomment the following line
// canvas->SaveAs("Hyy_fit_plot.pdf"); // pdf image

Now finally with our histogram, we can see a clear peak around the Higgs mass of 125 GeV. This means that we have successfully captured and visualized the Higgs boson! This is actually a significant result, and is very close to the [result reported by ATLAS](https://journals.aps.org/prd/abstract/10.1103/PhysRevD.98.052005) on the same decay process, which you can compare with below!

<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/13-TeV-examples/uproot_python/images/ImagesHiggs/HyyPaper.png?raw=1" style="width:50%"></CENTER>

While our main task may be done, there's still more to explore. Here are some additional tasks you can try with this notebook:

- Check how many events are being thrown away by each selection in the analysis
- Add more selections from the Higgs boson discovery paper or the Higgs decay to photons measurement paper
- Experiment with the functions used for signal and background fitting
- Adapt your favorite analysis to C++, whether it be one from Open Data, or one of your own!

<div class="alert alert-block alert-info">
We welcome your feedback on this notebook or any of our other materials! Please <a href="https://forms.gle/zKBqS1opAHHemv9U7">fill out this survey</a> to let us know how we're doing, and you can enter a raffle to win some <a href="https://atlas-secretariat.web.cern.ch/merchandise">ATLAS merchandise</a>!
</div>
